# Explicación del Código `main.py`

Este cuaderno explica paso a paso el funcionamiento del servicio de reentrenamiento y predicción de stock implementado en `main.py`.

## 1. Importaciones y Configuración

Aquí se importan las librerías necesarias: Flask para la API, TensorFlow/Keras para el modelo LSTM, Pandas/Numpy para manipulación de datos, y otras utilidades.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Input
from datetime import datetime, timedelta
from agent import preguntar_al_agente

@tf.keras.utils.register_keras_serializable()
def weighted_huber_loss(y_true, y_pred):
    return tf.keras.losses.huber(y_true, y_pred)

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 100 * 1024 * 1024 # 100MB limit

# Configuración de rutas
BASE_DIR = os.path.dirname(os.path.abspath('main.py'))
DATA_FILE = os.path.join(BASE_DIR, 'data', 'practica_completo.csv')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
MODEL_FILE = os.path.join(MODEL_DIR, 'model.keras')
SCALER_FILE = os.path.join(MODEL_DIR, 'scaler.joblib')

os.makedirs(os.path.dirname(DATA_FILE), exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## 2. Procesamiento de Datos

La función `procesar_dataframe` limpia y prepara los datos. Convierte fechas, ordena, rellena valores nulos y genera características adicionales (feature engineering) como día de la semana, mes, etc.

In [ ]:
FEATURES = ['salida', 'dia_semana', 'mes', 'fin_semana', 'feriado', 'quantity_on_hand', 'unit_cost']
VENTANA = 7 

def procesar_dataframe(df):
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    df = df.dropna(subset=['created_at'])
    df = df.sort_values('created_at')
    
    # feature engineering
    df['dia_semana'] = df['created_at'].dt.weekday
    df['mes'] = df['created_at'].dt.month
    df['fin_semana'] = df['created_at'].dt.weekday.apply(lambda x: 1 if x >= 5 else 0)
    df['feriado'] = 0 
    
    df = df.fillna(0)
    
    for col in FEATURES:
        if col not in df.columns:
            df[col] = 0
            
    return df

## 3. Entrenamiento del Modelo

La función `preparar_y_entrenar` es el núcleo del aprendizaje. Realiza los siguientes pasos:
1. Carga y procesa los datos.
2. Escala los valores entre 0 y 1.
3. Crea ventanas de tiempo (secuencias) para el modelo LSTM.
4. Carga el modelo existente o crea uno nuevo si falla la carga.
5. Entrena el modelo y lo guarda.

In [ ]:
def preparar_y_entrenar():
    print("--- Iniciando proceso de entrenamiento ---")
    if not os.path.exists(DATA_FILE):
        raise FileNotFoundError("Falta el archivo CSV")
    
    # 1 cargar
    df = pd.read_csv(DATA_FILE)
    df_procesado = procesar_dataframe(df)
    dataset_values = df_procesado[FEATURES].values
    
    # 2 escalar
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(dataset_values)
    joblib.dump(scaler, SCALER_FILE)
    print("Scaler guardado.")
    
    # 3 crear ventanas
    X, y = [], []
    limit_start = 0
    if len(scaled_data) > 5000:
        limit_start = len(scaled_data) - 5000
        
    for i in range(max(VENTANA, limit_start), len(scaled_data)):
        X.append(scaled_data[i-VENTANA:i]) 
        y.append(scaled_data[i, 0])       
        
    X, y = np.array(X), np.array(y)
    print(f"Entrenando con {len(X)} muestras...")
    
    # 4 CARGAR modelo existente o CREAR uno nuevo si falla
    try:
        print("DEBUG: VERSION 3.1 - INTENTANDO CARGAR MODELO...")
        model = tf.keras.models.load_model(MODEL_FILE, custom_objects={'weighted_huber_loss': weighted_huber_loss})
        
        # Validar compatibilidad
        dummy_input = np.zeros((1, X.shape[1], X.shape[2]))
        model.predict(dummy_input, verbose=0)
        print("Modelo cargado y validado correctamente.")
        
    except Exception as e:
        print(f"No se pudo cargar/validar el modelo: {e}. BORRANDO Y CREANDO UNO NUEVO.")
        if os.path.exists(MODEL_FILE):
            os.remove(MODEL_FILE)

        model = Sequential()
        model.add(Input(shape=(X.shape[1], X.shape[2]))) 
        model.add(LSTM(50, return_sequences=True))
        model.add(LSTM(50, return_sequences=False))
        model.add(Dense(25))
        model.add(Dense(1)) 
        model.compile(optimizer='adam', loss=weighted_huber_loss)
    
    # Entrenar
    model.fit(X, y, batch_size=32, epochs=1, verbose=1)
    
    model.save(MODEL_FILE)
    print("Modelo guardado exitosamente.")
    return len(df)

## 4. Endpoints de la API

Se definen las rutas de Flask para interactuar con el servicio.

### /api/v1/retrain
Recibe nuevos datos, los agrega al dataset y reentrena el modelo.

In [ ]:
@app.route('/api/v1/retrain', methods=['POST'])
def retrain():
    try:
        new_data = request.get_json().get('data')
        
        if new_data:
            df_new = pd.DataFrame(new_data)
            if os.path.exists(DATA_FILE):
                df_new.to_csv(DATA_FILE, mode='a', header=False, index=False)
            else:
                df_new.to_csv(DATA_FILE, mode='w', header=True, index=False)
            
        total = preparar_y_entrenar()
        
        return jsonify({"mensaje": "Entrenamiento completado", "total_registros": total})
    except Exception as e:
        print(e)
        return jsonify({"error": str(e)}), 500

### /api/v1/predict
Realiza predicciones de ventas y stock futuro para un producto específico. Utiliza un bucle recursivo para predecir día a día, actualizando el stock estimado en cada paso.

In [ ]:
@app.route('/api/v1/predict', methods=['POST'])
def predict():
    try:
        if not os.path.exists(SCALER_FILE):
            return jsonify({"error": "Primero debes entrenar el modelo"}), 400
            
        # 1 recibir datos
        content = request.json
        prod_id = content.get('product_id')
        target_date_str = content.get('date') # Formato: "YYYY-MM-DD"
        
        # cargar herramientas
        print("Cargando modelo para predict...")
        model = load_model(MODEL_FILE, custom_objects={'weighted_huber_loss': weighted_huber_loss})
        
        if len(model.inputs) != 1:
             raise ValueError(f"Modelo incompatible: espera {len(model.inputs)} entradas, se requiere 1.")
        scaler = joblib.load(SCALER_FILE)
        
        # cargar y procesar datos
        df = pd.read_csv(DATA_FILE)
        df = procesar_dataframe(df)
        if not prod_id:
            if not df.empty:
                latest_dates = df.groupby('product_id')['created_at'].max()
                prod_id = int(latest_dates.idxmax())
            else:
                return jsonify({"error": "No hay datos en la base de datos"}), 400
        
        df_prod = df[df['product_id'] == int(prod_id)].sort_values('created_at')
    
        # 2 determinar hasta cuándo predecir
        last_real_date = df_prod['created_at'].iloc[-1]
        
        days_to_predict = 1
        target_date_obj = last_real_date + timedelta(days=1)
        
        if target_date_str:
            target_date_obj = pd.to_datetime(target_date_str)
            days_diff = (target_date_obj - last_real_date).days
            days_to_predict = days_diff

        # 3 bucle recursivo
        current_window = df_prod[FEATURES].values[-VENTANA:]
        
        predicted_sales = 0
        new_stock = 0
        
        for i in range(days_to_predict):
            window_scaled = scaler.transform(current_window)
            X_input = np.array([window_scaled])
            
            pred_scaled = model.predict(X_input, verbose=0)
            
            dummy = np.zeros((1, len(FEATURES)))
            dummy[0, 0] = pred_scaled[0][0]
            pred_sales_unscaled = scaler.inverse_transform(dummy)[0][0]
            
            if i == days_to_predict - 1:
                predicted_sales = pred_sales_unscaled
            
            next_day_date = last_real_date + timedelta(days=i+1)
            
            last_stock = current_window[-1][5]
            new_stock = max(0, last_stock - pred_sales_unscaled)
            
            new_row = [
                pred_sales_unscaled,
                next_day_date.weekday(),
                next_day_date.month,
                1 if next_day_date.weekday() >= 5 else 0,
                0,
                new_stock,
                current_window[-1][6]
            ]
            
            current_window = np.vstack([current_window[1:], new_row])
        
        return jsonify({
            "product_id": prod_id,
            "target_date": target_date_obj.strftime('%Y-%m-%d'),
            "days_ahead": days_to_predict,
            "prediccion_salida": float(predicted_sales),
            "stock_estimado": float(new_stock)
        })
        
    except Exception as e:
        return jsonify({"error": str(e)}), 500

### /api/v1/chat
Integra un agente externo para responder preguntas.

In [ ]:
@app.route('/api/v1/chat', methods=['POST'])
def chat_agent():
    content = request.json
    pregunta = content.get('pregunta')
    if not pregunta: return jsonify({"error": "Falta pregunta"}), 400
    
    respuesta = preguntar_al_agente(pregunta)

    return jsonify({"respuesta_ia": respuesta})

## 5. Ejecución

Inicia el servidor Flask.

In [ ]:
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=1919, debug=True)

## 6. Despliegue en Google Cloud Run

Para llevar este servicio a producción, utilizamos Google Cloud Run, que nos permite ejecutar contenedores sin servidor de manera escalable.

### Pasos principales:
1. **Dockerización**: Se utiliza un `Dockerfile` para empaquetar la aplicación, instalando Python, las dependencias (TensorFlow, Flask, etc.) y copiando el código fuente.
2. **Build**: Se construye la imagen del contenedor y se sube a Google Container Registry (GCR) o Artifact Registry.
3. **Deploy**: Se despliega la imagen en Cloud Run, exponiendo el puerto definido (1919 en este caso, aunque Cloud Run suele inyectar la variable PORT).

**Comandos típicos de despliegue:**

In [ ]:
# Ejemplo de construcción y subida de la imagen
# gcloud builds submit --tag gcr.io/TU_PROYECTO/stock-retrain-service

# Ejemplo de despliegue en Cloud Run
# gcloud run deploy stock-retrain-service --image gcr.io/TU_PROYECTO/stock-retrain-service --platform managed --allow-unauthenticated

## 7. Desafíos de Infraestructura: Consumo de Memoria RAM con TensorFlow

Un punto crítico encontrado durante el desarrollo y despliegue fue el manejo de recursos del sistema, específicamente la memoria RAM.

### El Problema
TensorFlow es un framework robusto pero pesado. Al inicializar y cargar modelos (especialmente redes neuronales recurrentes como LSTM), requiere reservar una cantidad significativa de memoria para:
- Cargar las librerías dinámicas de C++ (CUDA, cuDNN si aplica, etc.).
- Construir el grafo de computación.
- Almacenar los pesos del modelo y los estados intermedios.

Las instancias por defecto de muchos servicios serverless (incluyendo configuraciones básicas de Cloud Run) a menudo inician con **512MB** de RAM. Esto resultó ser insuficiente para levantar el servicio con TensorFlow, provocando errores de tipo **"Out of Memory" (OOM)** o reinicios silenciosos del contenedor (CrashLoopBackOff) justo al intentar ejecutar `load_model`.

### La Solución
Para solucionar este inconveniente y asegurar la estabilidad del servicio:
1. **Incremento de Recursos**: Fue necesario configurar la instancia de Cloud Run para utilizar un mínimo de **2GB de RAM**.
2. **Gestión de Caché**: Se implementaron validaciones para asegurar que el modelo solo se cargue cuando es necesario o mantenerlo en memoria de manera eficiente, aunque el aumento de RAM fue la solución definitiva.

Esta consideración es vital para cualquier servicio de ML en producción: los requerimientos de inferencia y entrenamiento suelen superar los de una API web estándar.